In [1]:
import numpy as np
from veripulse.pulse import run_grape_si, run_grape, PulseConfig, print_pulse_config
from veripulse.gates import rx, rhox, pack_subspace_states, extract_subspace_states, Qobj, operator_to_vector, vector_to_operator
from veripulse.sdp import choi_optimise_secret_indep, calc_secret_indep

# Configurations and target states

In [3]:
conf1 = PulseConfig() # default
print_pulse_config(conf1)

Parameter                      Value Description
─────────────────────────────────────────────
omega_drift                1.000e+07  Hz
T2_star                    2.000e-06  s
drive_error                3.000e-02  
detuning                   0.000e+00  normalised
evo_time                   3.000e-07  s
num_tslots                       100  
amp_lbound                -1.000e+00  
amp_ubound                 1.000e+00  
fid_err_targ               1.000e-08  
max_iter                        1000  
max_wall_time              1.200e+02  s
init_pulse_type                  RND  
fid_params                        {}  


In [4]:
# number of states 
K = 8 
# initial states with a little noise 
err= 1e-3
rho_init = Qobj([[1-err,0],[0,err]])
# target states 
rho_targets = [Qobj(rhox(i*np.pi/4)) for i in range(K)]

In [5]:
rho_init

Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=Dense, isherm=True
Qobj data =
[[0.999 0.   ]
 [0.    0.001]]

# 1 Regular GRAPE

In [6]:
# GRAPE optimisation part
result1 = []
for rho_targ in rho_targets:
    res = run_grape(operator_to_vector(rho_init), operator_to_vector(rho_targ))
    result1.append(res)

In [7]:
# Check optimisation results
for res in result1:
    print("fid err:",res.fid_err)

fid err: 1.7773265830926223e-06
fid err: 1.6796787132300867e-05
fid err: 0.00011288021796972308
fid err: 1.3877262558850032e-05
fid err: 1.508178608778044e-05
fid err: 4.56935265840393e-05
fid err: 3.6864854783447926e-05
fid err: 3.0379660230786623e-06


**Calculate secret independence**

In [8]:
# state formatting
rho_fin_res1 = [vector_to_operator(res.evo_full_final).full() for res in result1]
rho_targ_np = [rho.full() for rho in rho_targets] 

In [9]:
res1_si = choi_optimise_secret_indep(rho_targ_np, rho_fin_res1)

In [10]:
res1_si.objective

np.float64(0.0033738649123231626)

# 2 GRAPE with noise averaging methode

In [13]:
# packing states 
K = 8
vRho_init, vRho_target, U_big = pack_subspace_states(
    rotations=[rx(j * np.pi / 4) for j in range(K)],
    rho_init=rho_init,
)

In [15]:
conf2 = PulseConfig(fid_err_targ=1e-10, max_iter=10000000, max_wall_time=10000)
print_pulse_config(conf2)

Parameter                      Value Description
─────────────────────────────────────────────
omega_drift                1.000e+07  Hz
T2_star                    2.000e-06  s
drive_error                3.000e-02  
detuning                   0.000e+00  normalised
evo_time                   3.000e-07  s
num_tslots                       100  
amp_lbound                -1.000e+00  
amp_ubound                 1.000e+00  
fid_err_targ               1.000e-10  
max_iter                    10000000  
max_wall_time                  10000  s
init_pulse_type                  RND  
fid_params                        {}  


In [17]:
result2 = run_grape_si(vRho_init, vRho_target, U_big, lam=0.01, config=conf2)

Secret Independence =  0.38389255729967864
fidelity error =  0.00023445837308291184
Secret Independence =  0.38361245586996173
fidelity error =  0.00023448495736594008
Secret Independence =  0.38333172084593914
fidelity error =  0.00023451165474272745
Secret Independence =  0.3822057471434125
fidelity error =  0.00023461791487083182
Secret Independence =  0.3782104961145355
fidelity error =  0.0002349847096143299
Secret Independence =  0.37791869578216347
fidelity error =  0.0002350135330154727
Secret Independence =  0.3767485183897127
fidelity error =  0.0002351282694342632
Secret Independence =  0.37202190075681896
fidelity error =  0.00023557830468913228
Secret Independence =  0.3643182069983484
fidelity error =  0.00023626883271585654
Secret Independence =  0.3639994085464732
fidelity error =  0.00023630276104429575
Secret Independence =  0.3627214088545284
fidelity error =  0.00023643781020535838
Secret Independence =  0.35756655156916073
fidelity error =  0.00023696738160706055
S

In [21]:
rho_fin_res2=extract_subspace_states(result2, K)

In [25]:
res2_si = choi_optimise_secret_indep(rho_targ_np, rho_fin_res2)

In [26]:
res2_si.objective

np.float64(0.002579687256024297)

In [28]:
res2_si.status

'optimal'